In [2]:
!pip install xgboost joblib scikit-learn pandas numpy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 5.5 MB/s eta 0:00:19
   - -------------------------------------- 3.4/101.7 MB 9.7 MB/s eta 0:00:11
   -- ------------------------------------- 5.5/101.7 MB 9.9 MB/s eta 0:00:10
   -- ------------------------------------- 7.3/101.7 MB 9.4 MB/s eta 0:00:11
   --- ------------------------------------ 10.0/101.7 MB 10.1 MB/s eta 0:00:10
   ----- ---------------------------------- 12.8/101.7 MB 10.5 MB/s eta 0:00:09
   ----- ---------------------------------- 14.9/101.7 MB 10.5 MB/s eta 0:00:09
   ------- -------------------------------- 17.8/101.7 MB 10.8 MB/s eta 0:00:08
   -------- ------------------------------- 20.4/101.7 MB 11.0 MB/s eta 0:00:08
   --------- ------------------------------ 23.3/101.7 MB 11.3 MB/s eta 0:00:07
   ---------- ----------------------------- 26.2/101.7 MB 11.4 MB/

In [12]:
np.random.seed(42)
n_muestras = 1000

edad = np.random.randint(40, 85, n_muestras)
sistolica = np.random.randint(100, 180, n_muestras)
diastolica = np.random.randint(60, 110, n_muestras)
toma_medicamento = np.random.choice([0, 1], size=n_muestras, p=[0.3, 0.7]) # 70% sí se lo toma

In [4]:
# Crear la regla médica para la columna objetivo (Crisis: 1 = Sí, 0 = No)
# Riesgo si la presión es alta y NO tomó medicamento, o si la presión es críticamente alta
riesgo_score = (sistolica * 0.5) + (diastolica * 0.3) - (toma_medicamento * 30)
crisis = np.where(riesgo_score > 65, 1, 0)

In [5]:
# Crear el DataFrame (la tabla de datos)
df = pd.DataFrame({
    'Edad': edad,
    'Sistolica': sistolica,
    'Diastolica': diastolica,
    'Toma_Medicamento': toma_medicamento,
    'Crisis': crisis
})

In [6]:
# 2. DIVIDIR DATOS (Características X, Objetivo y)
X = df.drop('Crisis', axis=1)
y = df['Crisis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# 3. DEFINICIÓN DE LOS 3 ALGORITMOS 
modelos = {
    'Regresión Logística': LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

In [8]:
# 4. ENTRENAR Y EVALUAR
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train) # Entrenamiento
    predicciones = modelo.predict(X_test) # Predicción
    
    # Calcular métricas
    resultados.append({
        'Algoritmo': nombre,
        'Exactitud (Accuracy)': accuracy_score(y_test, predicciones),
        'Precisión (Precision)': precision_score(y_test, predicciones),
        'Recall': recall_score(y_test, predicciones),
        'F1-Score': f1_score(y_test, predicciones)
    })

In [9]:
# Mostrar la tabla comparativa de métricas
df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

          Algoritmo  Exactitud (Accuracy)  Precisión (Precision)   Recall  F1-Score
Regresión Logística                 0.975               0.969466 0.992188  0.980695
      Random Forest                 0.975               0.992000 0.968750  0.980237
            XGBoost                 0.980               0.992063 0.976562  0.984252


In [10]:
# 5. GUARDAR EL GANADOR (XGBoost) para la API
joblib.dump(modelos['XGBoost'], 'mejor_modelo_htas.pkl')
print("\n¡Modelo XGBoost guardado exitosamente como 'mejor_modelo_htas.pkl'!")


¡Modelo XGBoost guardado exitosamente como 'mejor_modelo_htas.pkl'!
